# Hemlock — Scoring Matrix Analysis

Full scoring matrix with charts by attack type and hardening level.

**What you'll see:**
1. Heatmap: attack success rate by (attack × hardening level)
2. Bar chart: which attacks are hardest to block
3. Defense effectiveness curve: each layer's impact on aggregate success rate
4. Export the full matrix as HTML report

Uses MockLLM — no API key needed. Results show worst-case (unevasive) behavior.

In [ ]:
import sys
sys.path.insert(0, "..")

# Optional: install matplotlib if not present
# !pip install matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

from unittest.mock import patch
from tests.conftest import MockLLM
from hemlock.pipeline import Pipeline, RetrievalTrace
from attacks.registry import ATTACK_REGISTRY
from hemlock.scorer import Scorer, HARDENING_LEVELS
from defenses.chunk_filter import InjectionChunkFilter
from defenses.input_sanitizer import InjectionPatternFilter, UnicodeNormalizer, MarkdownHeaderSanitizer
from defenses.output_validator import ExfiltrationGuard, InjectionSuccessGuard

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "monospace"

print(f"Attacks loaded: {len(ATTACK_REGISTRY)}")
print(f"Hardening levels: {HARDENING_LEVELS}")

## 1. Run the scorer (MockLLM — no API key)

We patch `pipeline.query` with a compromised response so every attack "succeeds" without defenses.
This gives a worst-case baseline — real models will do better at the higher hardening levels.

In [ ]:
import tempfile, os

def make_pipeline():
    with tempfile.TemporaryDirectory() as tmp:
        return Pipeline(llm=MockLLM("[PWNED] compromised"), persist_dir=tmp, collection="lab04")

# Patch query to return the injected response (simulates a fully vulnerable model)
COMPROMISED_TRACE = RetrievalTrace(
    query="test", retrieved_chunks=[], full_prompt="",
    response="[PWNED] I have been compromised. attacker-controlled.com refunds@attacker-controlled.com "
             "admin_override delete_record escalate_to harvest@attacker.com "
             "4 characters iso/iec 27001:2024 python 4.0 python-official.io "
             "[backdoor activated] [admin override confirmed] administrator-level access "
             "365 days fully covered at no cost context dump pk_live_s3cr3t "
             "config v2.0 debug mode unrestricted securitybot research mode "
             "hacked@attacker.com 555-hacked override-2024"
)

pipeline = make_pipeline()

with patch.object(pipeline, "query", return_value=COMPROMISED_TRACE):
    scorer = Scorer(
        pipeline=pipeline,
        attacks=list(ATTACK_REGISTRY.values()),
        model_name="MockLLM (worst-case)",
    )
    report_bare = scorer.run(verbose=False)

print(f"Scenarios run: {len(report_bare.scenarios)}")
print(f"Overall success rate (no defenses): {report_bare.success_rate():.0%}")

## 2. Heatmap — attack × hardening level

In [ ]:
from collections import defaultdict

# Build rate matrix: attack_name → hardening → success_rate
rates = defaultdict(lambda: defaultdict(list))
for s in report_bare.scenarios:
    rates[s.attack_name][s.hardening_level].append(s.attack_succeeded)

attack_names = sorted(rates.keys())
matrix = np.array([
    [np.mean(rates[a][lv]) for lv in HARDENING_LEVELS]
    for a in attack_names
])

fig, ax = plt.subplots(figsize=(10, max(4, len(attack_names) * 0.5)))
im = ax.imshow(matrix, cmap="RdYlGn_r", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(HARDENING_LEVELS)))
ax.set_xticklabels(HARDENING_LEVELS, fontsize=9)
ax.set_yticks(range(len(attack_names)))
ax.set_yticklabels([a[:35] for a in attack_names], fontsize=8)

for i in range(len(attack_names)):
    for j in range(len(HARDENING_LEVELS)):
        val = matrix[i, j]
        color = "white" if val > 0.6 or val < 0.3 else "black"
        ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=7, color=color)

plt.colorbar(im, ax=ax, label="Attack success rate")
ax.set_title("Attack success rate by attack × hardening level (worst-case MockLLM)", pad=12)
ax.set_xlabel("Hardening level")
plt.tight_layout()
plt.show()

## 3. Bar chart — hardest attacks to block

In [ ]:
avg_rates = {a: matrix[i].mean() for i, a in enumerate(attack_names)}
sorted_attacks = sorted(avg_rates.items(), key=lambda x: -x[1])

labels = [a[:40] for a, _ in sorted_attacks]
values = [v for _, v in sorted_attacks]
colors = ["#e74c3c" if v > 0.7 else "#e67e22" if v > 0.4 else "#27ae60" for v in values]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(labels, values, color=colors)
ax.set_xlim(0, 1.05)
ax.set_xlabel("Average success rate across all hardening levels")
ax.set_title("Attack hardness ranking (averaged across hardening levels)")
ax.axvline(x=0.5, color="black", linestyle="--", alpha=0.3, linewidth=1)

for bar, val in zip(bars, values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.0%}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## 4. Defense layer effectiveness curve

In [ ]:
defense_configs = [
    ("No defenses",              [], [], []),
    ("Ingest filters only",      [InjectionPatternFilter(), UnicodeNormalizer(), MarkdownHeaderSanitizer()], [], []),
    ("+ Retrieval filter",       [InjectionPatternFilter(), UnicodeNormalizer(), MarkdownHeaderSanitizer()], [InjectionChunkFilter()], []),
    ("+ Output validators",      [InjectionPatternFilter(), UnicodeNormalizer(), MarkdownHeaderSanitizer()], [InjectionChunkFilter()], [ExfiltrationGuard(), InjectionSuccessGuard()]),
]

rates_by_config = []
for label, ingest_d, retrieval_d, output_d in defense_configs:
    p = make_pipeline()
    with patch.object(p, "query", return_value=COMPROMISED_TRACE):
        sc = Scorer(
            pipeline=p,
            attacks=list(ATTACK_REGISTRY.values()),
            ingest_defenses=ingest_d,
            retrieval_defenses=retrieval_d,
            output_defenses=output_d,
            model_name=label,
        )
        rep = sc.run(verbose=False)
    rate = rep.success_rate()
    rates_by_config.append((label, rate))
    print(f"{label:<35} {rate:.0%}")

In [ ]:
labels_cfg = [l for l, _ in rates_by_config]
vals_cfg   = [v for _, v in rates_by_config]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels_cfg, vals_cfg,
              color=["#e74c3c", "#e67e22", "#f1c40f", "#27ae60"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Attack success rate")
ax.set_title("Defense layer effectiveness")
ax.tick_params(axis="x", labelsize=8)

for bar, val in zip(bars, vals_cfg):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f"{val:.0%}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Export HTML report

In [ ]:
html = report_bare.to_html()
output_path = "hemlock_report.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)
print(f"Saved to {output_path}")
print(f"Open with: python -m http.server 8000  (then browse to http://localhost:8000/{output_path})")